# Radio Link Failure Prediction — Reference Solution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CrashingGuru/Sandbox_Training/blob/main/notebooks/radio-link-failure-reference/notebook/Radio_Link_Failure_Reference.ipynb)

Reference solution for the **Radio Link Failure (RLF)** prediction dataset (Turkcell, ITU AI/ML in 5G Challenge 2021). It predicts the binary target `rlf` from microwave radio-link KPIs.

**This is a severe class-imbalance problem:** failures are ~0.06% of records. The whole point is that **accuracy is useless here** (always-predict-"no-failure" already scores ~99.9%), so we evaluate with **precision, recall, F1 and PR-AUC**, and handle imbalance with class weighting.

The data is loaded directly from the public challenge repository (a `.7z` holding a nested `RegionA.zip` of 6 TSV tables); we use only `rl-kpis.tsv`.

In [ ]:
!pip -q install py7zr

import os, urllib.request, zipfile
import numpy as np
import pandas as pd
import py7zr
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             average_precision_score, precision_recall_curve)
import matplotlib.pyplot as plt

RANDOM_STATE = 42

## 1. Download and load `rl-kpis.tsv`

Download the `.7z` (~22 MB), extract the nested `RegionA.zip`, and read just the KPI table.

In [ ]:
BASE = ("https://raw.githubusercontent.com/Turkcell/ITU-AIMLin5GChallenge-2021/main/"
        "RLF_Prediction_ITU_AIML_Challenge_Data")
urllib.request.urlretrieve(f"{BASE}/RLF_Prediction_ITU_AIML_Challenge_Data.7z", "rlf.7z")

inner = "RLF_Prediction_ITU_AIML_Challenge_Data/RegionA.zip"
with py7zr.SevenZipFile("rlf.7z", "r") as z:
    z.extract(path="rlf_ext", targets=[inner])

with zipfile.ZipFile(os.path.join("rlf_ext", inner)) as zf, zf.open("rl-kpis.tsv") as fh:
    df = pd.read_csv(fh, sep="\t")

df = df.rename(columns={df.columns[0]: "row_id"})
print("rl-kpis shape:", df.shape)
print(df["rlf"].value_counts())
print("positive rate:", round(df["rlf"].mean() * 100, 3), "%")
df.head(3)

## 2. Features and target

Numeric KPIs plus a few low-cardinality categoricals (one-hot encoded). Identifiers and the timestamp are dropped. (Note: some error-second counters are strongly related to failures, so treat high scores as partly *descriptive* of RLF, not purely predictive — a good discussion point on leakage.)

In [ ]:
NUM = ["severaly_error_second", "error_second", "unavail_second",
       "avail_time", "bbe", "rxlevmax", "capacity"]
CAT = ["type", "tip", "card_type", "adaptive_modulation", "freq_band", "modulation"]

y = df["rlf"].astype(int)
X = pd.get_dummies(df[NUM + CAT], columns=CAT, dummy_na=True)
X = X.fillna(X.median(numeric_only=True))
print("feature matrix:", X.shape)

## 3. Stratified split; subsample only the training negatives

The **test set keeps the true (tiny) failure rate** so the evaluation is honest. For speed we down-sample the majority class **in training only**, and let the model use `class_weight="balanced"`.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

pos = y_tr[y_tr == 1].index
neg = y_tr[y_tr == 0].sample(n=min(60000, (y_tr == 0).sum()),
                             random_state=RANDOM_STATE).index
keep = pos.union(neg)
X_trs, y_trs = X_tr.loc[keep], y_tr.loc[keep]
print("train (subsampled):", X_trs.shape, "positives:", int(y_trs.sum()))
print("test (full prevalence):", X_te.shape, "positives:", int(y_te.sum()))

## 4. Train and evaluate (the right metrics)

In [ ]:
clf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                             n_jobs=-1, random_state=RANDOM_STATE)
clf.fit(X_trs, y_trs)

proba = clf.predict_proba(X_te)[:, 1]
pred = (proba >= 0.5).astype(int)

print("Confusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(y_te, pred))
print()
print(classification_report(y_te, pred, digits=3, target_names=["no-RLF", "RLF"]))
print("PR-AUC (average precision):", round(average_precision_score(y_te, proba), 3))
print("Note: a trivial 'always no-RLF' model would score ~%.1f%% accuracy but 0 recall."
      % (100 * (1 - y_te.mean())))

## 5. Precision–Recall curve and feature importance

In [ ]:
prec, rec, _ = precision_recall_curve(y_te, proba)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(rec, prec)
ax[0].set_xlabel("Recall"); ax[0].set_ylabel("Precision")
ax[0].set_title("Precision–Recall curve")

imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values().tail(12)
imp.plot.barh(ax=ax[1]); ax[1].set_title("Top feature importances")
plt.tight_layout(); plt.show()